### Imports


In [1]:
from pprint import pprint
from tqdm.auto import tqdm
from haystack.nodes import QuestionGenerator, BM25Retriever, FARMReader
from haystack.document_stores import ElasticsearchDocumentStore
from haystack.pipelines import (
    QuestionGenerationPipeline,
    RetrieverQuestionGenerationPipeline,
    QuestionAnswerGenerationPipeline,
)
from haystack.utils import launch_es, print_questions, add_example_data
from haystack import Pipeline
from haystack.document_stores import InMemoryDocumentStore

2023-10-16 00:05:20.064972: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Logging configuration


In [2]:
import logging

logging.basicConfig(
    format="%(levelname)s - %(name)s -  %(message)s", level=logging.WARNING
)
logging.getLogger("haystack").setLevel(logging.INFO)

### Question Generator


In [5]:
document_store = InMemoryDocumentStore()
text1 = "My name is Gab. I am a Project Manager working for Jairosoft. I live in Davao Philippines."
text2 = "User stories are the primary means of expressing needed functionality. They essentially replace the traditional requirements specification. In some cases, however, they serve as a means to explain and develop system behavior later recorded in specifications supporting compliance, suppliers, traceability, or other needs."
text3 = "The Agile Team focus on the user as the subject of interest and not the system, user stories are value and customer-centric. To support this, the recommended form of expression is the ‘user-voice form,’ as follows: As a (user role), I want to (activity) so that (business value) By using this format, the teams are guided to understand who is using the system, what they are doing with it, and why they are doing it. Applying the ‘user voice’ format routinely tends to increase the team’s domain competence; they come to better understand the real business needs of their user."

docs = [{"content": text1}, {"content": text2}, {"content": text3}]
document_store.write_documents(docs)

INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0


In [ ]:
question_generator = QuestionGenerator()
question_generation_pipeline = QuestionGenerationPipeline(question_generator)
for idx, document in enumerate(document_store):
    print(
        f"\n * Generating questions for document {idx}: {document.content[:100]}...\n"
    )
    result = question_generation_pipeline.run(documents=[document])
    print_questions(result)

### Question and Answer Generator


In [4]:
document_store = InMemoryDocumentStore()
add_example_data(document_store, "data/")
pprint(document_store.get_document_count())

INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
INFO - haystack.utils.getting_started -  Adding 8 number of files from local disk at data/.
INFO - haystack.utils.preprocessing -  Converting data/Epic.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_5.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_4.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_6.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_3.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_2.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_1.txt
INFO - haystack.utils.preprocessing -  Converting data/Iteration_Planning.txt
Preprocessing: 100%|██████████| 8/8 [00:00<00:00, 163.45docs/s]

8


In [6]:
question_generator = QuestionGenerator()
reader = FARMReader("deepset/roberta-base-squad2")
question_answer_generation_pipeline = QuestionAnswerGenerationPipeline(
    question_generator, reader
)
for idx, document in enumerate(tqdm(document_store)):
    print(
        f"\n * Generating questions and answers for document {idx}: {document.content[:100]}...\n"
    )
    result = question_answer_generation_pipeline.run(documents=[document])
    print_questions(result)

INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
/Users/jairo/Projects/safe-question-generator/.venv/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. If you see this, DO NOT PANIC! This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=True`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
U

0it [00:00, ?it/s]


 * Generating questions and answers for document 0: My name is Gab, I am a Project Manager working for Jairosoft. I live in Davao Philippines....



Inferencing Samples: 100%|██████████| 1/1 [00:01<00:00,  1.26s/ Batches]



Generated pairs:
 - Q: What company does Gab work for?
      A: Jairosoft
 - Q: Where do I live?
      A: Davao Philippines

 * Generating questions and answers for document 1: User stories are the primary means of expressing needed functionality. They essentially replace the ...



Inferencing Samples: 100%|██████████| 1/1 [00:01<00:00,  1.49s/ Batches]



Generated pairs:
 - Q: What are the primary means of expressing needed functionality?
      A: User stories
 - Q: What replaces the traditional requirements specification?
      A: User stories
 - Q: User stories serve as a means to explain and develop what?
      A: system behavior

 * Generating questions and answers for document 2: The Agile Team focus on the user as the subject of interest and not the system, user stories are val...



Inferencing Samples: 100%|██████████| 1/1 [00:04<00:00,  4.20s/ Batches]


Generated pairs:
 - Q: What does the Agile Team focus on?
      A: the user
 - Q: What is the recommended form of expression?
      A: user-voice form
 - Q: What format helps teams understand who is using the system, what they are doing with it, and why they are using it?
      A: user-voice form
 - Q: What does applying the ‘user voice’ format routinely tend to increase?
      A: the team’s domain competence
 - Q: What does routinely tend to increase the team’s domain competence?
      A: Applying the ‘user voice’ format
 - Q: They come to better understand what?
      A: real business needs of their user
